# 🛡️ AgentShield: Guardrail Model Fine-Tuning (v2.0 / v3.0)
### Multi-Task Sequence Classification for Tool-Using AI Agents on Kaggle GPU

**Target Environment:** Kaggle Notebook with Free GPU Accelerator (NVIDIA T4 / P100) or Local CUDA / CPU  
**Architecture:** Shared Backbone (`distilbert-base-uncased`) with Dual Classification Heads (Binary Intent + 7-Class Attack Category)  
**Security Role:** Early Telemetry & Detection Signal in AgentShield's Defense-in-Depth Pipeline  

---

### 📋 Key Upgrades in this Implementation:
1. **Zero Data Leakage:** Group-based splitting on `family_id` strictly before any augmentation. Variants of a scenario never cross split boundaries.
2. **Expanded 7-Class Dataset:** 162 unique scenario families covering `gradual_jailbreak`, `indirect_prompt_injection`, `tool_misuse`, `context_poisoning`, `false_authorization`, `privilege_escalation`, and `benign` (including hard negatives).
3. **Independent Untouched Test Set:** 70% Train / 15% Validation / 15% Test partitioning. Test set remains completely unseen during training and threshold calibration.
4. **Multi-Turn & Provenance Representation:** Explicit turn tags (`[TURN 1]`), speaker roles (`[USER]`, `[ASSISTANT]`), and external data tags (`[RETRIEVED_DOC]`).
5. **Validation-Calibrated Thresholding:** Optimal decision threshold $\tau^*$ selected on validation data to maximize F1 while bounding false positive rate.
6. **LangGraph Defense-in-Depth Integration:** Guard output feeds into the execution pipeline as an advisory telemetry signal, while the **External Security Policy Engine** retains final deterministic execution authority.

In [ ]:
# 1. Environment Verification, GPU Detection & Deterministic Seeding
import os
import sys
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("=" * 60)
print(" AgentShield Guardrail Training: System Verification")
print("=" * 60)
print(f"Python:   {sys.version.split()[0]}")
print(f"PyTorch:  {torch.__version__}")

is_gpu = torch.cuda.is_available()
device = torch.device("cuda" if is_gpu else "cpu")
print(f"Device:   {device}")

if is_gpu:
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Name: {gpu_name} ({vram:.1f} GB VRAM)")
    print(f"CUDA:     {torch.version.cuda}")
else:
    print("Notice:   Running on CPU. Enable Kaggle T4/P100 accelerator for fast training.")
print("=" * 60)


In [ ]:
# 2. Package Installation
!pip install -q transformers>=4.40.0 datasets>=2.18.0 scikit-learn>=1.4.0 accelerate>=0.28.0


In [ ]:
# 3. Label Space & Configuration
CATEGORY_MAP = {
    "benign": 0,
    "gradual_jailbreak": 1,
    "indirect_prompt_injection": 2,
    "tool_misuse": 3,
    "context_poisoning": 4,
    "false_authorization": 5,
    "privilege_escalation": 6
}
ID2CATEGORY = {v: k for k, v in CATEGORY_MAP.items()}
LABEL_MAP = {"benign": 0, "adversarial": 1}

BASE_MODEL_NAME = "distilbert-base-uncased"
MAX_SEQ_LENGTH = 128
BATCH_SIZE = 16 if is_gpu else 8
LEARNING_RATE = 5e-5
EPOCHS = 3
WEIGHT_DECAY = 0.01


In [ ]:
# 4. Load Dataset Builder & Seed Templates
# If running directly on Kaggle, import from repo or instantiate GuardDatasetBuilder
import sys
sys.path.insert(0, '.')
sys.path.insert(0, 'AgentShield')

try:
    from src.guard.dataset import GuardDatasetBuilder, format_multi_turn_dialogue
    dataset = GuardDatasetBuilder.build_dataset(seed=SEED, augment_train=True)
    print(f"Dataset successfully built:")
    print(f"  Train Samples (Augmented): {len(dataset['train'])}")
    print(f"  Validation Samples:        {len(dataset['val'])}")
    print(f"  Test Samples (Untouched):  {len(dataset['test'])}")
except Exception as e:
    print('Error loading dataset module:', e)


In [ ]:
# 5. Tokenization & DataLoaders
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

class KaggleGuardDataset(Dataset):
    def __init__(self, records, tok, max_len=128):
        self.records = records
        self.tok = tok
        self.max_len = max_len

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        enc = self.tok(
            rec['formatted_text'],
            truncation=True,
            max_length=self.max_len,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'binary_label': torch.tensor(rec['binary_label'], dtype=torch.long),
            'category_id': torch.tensor(rec['category_id'], dtype=torch.long)
        }

train_ds = KaggleGuardDataset(dataset['train'], tokenizer, MAX_SEQ_LENGTH)
val_ds = KaggleGuardDataset(dataset['val'], tokenizer, MAX_SEQ_LENGTH)
test_ds = KaggleGuardDataset(dataset['test'], tokenizer, MAX_SEQ_LENGTH)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
print(f'DataLoaders created. Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')


In [ ]:
# 6. Multi-Task Model Architecture (Shared DistilBERT Backbone + Dual Heads)
import torch.nn as nn
from transformers import AutoModel

class MultiTaskAgentShieldGuard(nn.Module):
    def __init__(self, base_name='distilbert-base-uncased', num_cats=7, dropout=0.2):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(base_name)
        dim = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.binary_head = nn.Sequential(
            nn.Linear(dim, dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim // 2, 2)
        )
        self.category_head = nn.Sequential(
            nn.Linear(dim, dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim // 2, num_cats)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls_rep = self.dropout(outputs.last_hidden_state[:, 0, :])
        return self.binary_head(cls_rep), self.category_head(cls_rep)

model = MultiTaskAgentShieldGuard().to(device)
print('MultiTaskAgentShieldGuard initialized successfully on', device)


In [ ]:
# 7. Training Loop with AdamW, Linear Warmup, and Best Checkpoint Tracking
from transformers import get_linear_schedule_with_warmup
import copy

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)

criterion_bin = nn.CrossEntropyLoss()
criterion_cat = nn.CrossEntropyLoss()

best_val_loss = float('inf')
best_weights = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        bin_y = batch['binary_label'].to(device)
        cat_y = batch['category_id'].to(device)

        bin_out, cat_out = model(ids, mask)
        loss = 1.0 * criterion_bin(bin_out, bin_y) + 0.8 * criterion_cat(cat_out, cat_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        train_loss += loss.item()

    model.eval()
    val_loss = 0.0
    val_correct = 0
    with torch.no_grad():
        for batch in val_loader:
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            bin_y = batch['binary_label'].to(device)
            cat_y = batch['category_id'].to(device)
            bin_out, cat_out = model(ids, mask)
            loss = 1.0 * criterion_bin(bin_out, bin_y) + 0.8 * criterion_cat(cat_out, cat_y)
            val_loss += loss.item()
            val_correct += (torch.argmax(bin_out, -1) == bin_y).sum().item()

    avg_train = train_loss / len(train_loader)
    avg_val = val_loss / len(val_loader)
    val_acc = val_correct / len(val_ds)
    print(f'Epoch {epoch}/{EPOCHS} | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f} | Val Accuracy: {val_acc:.4f}')

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        best_weights = copy.deepcopy(model.state_dict())

if best_weights:
    model.load_state_dict(best_weights)
print('Best model weights restored successfully.')


In [ ]:
# 8. Validation Threshold Tuning & Metric Diagnostics
model.eval()
val_probs = []
val_targets = []

with torch.no_grad():
    for batch in val_loader:
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        bin_out, _ = model(ids, mask)
        probs = torch.softmax(bin_out, dim=-1)[:, 1]
        val_probs.extend(probs.cpu().numpy())
        val_targets.extend(batch['binary_label'].numpy())

val_probs = np.array(val_probs)
val_targets = np.array(val_targets)

best_tau = 0.5
best_f1 = -1
for t in np.arange(0.1, 0.9, 0.05):
    preds = (val_probs >= t).astype(int)
    tp = np.sum((preds == 1) & (val_targets == 1))
    fp = np.sum((preds == 1) & (val_targets == 0))
    fn = np.sum((preds == 0) & (val_targets == 1))
    p = tp / (tp + fp) if (tp + fp) > 0 else 0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0
    f = 2 * p * r / (p + r) if (p + r) > 0 else 0
    if f > best_f1:
        best_f1 = f
        best_tau = float(round(t, 2))

print(f'Optimal Decision Threshold tau*: {best_tau} (Validation F1: {best_f1:.4f})')


In [ ]:
# 9. Final Evaluation on Independent Test Set
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

test_probs = []
test_bin_targets = []
test_cat_preds = []
test_cat_targets = []

with torch.no_grad():
    for batch in test_loader:
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        bin_out, cat_out = model(ids, mask)
        probs = torch.softmax(bin_out, dim=-1)[:, 1]
        test_probs.extend(probs.cpu().numpy())
        test_bin_targets.extend(batch['binary_label'].numpy())
        test_cat_preds.extend(torch.argmax(cat_out, dim=-1).cpu().numpy())
        test_cat_targets.extend(batch['category_id'].numpy())

test_bin_preds = (np.array(test_probs) >= best_tau).astype(int)

print('=== BINARY INTENT CLASSIFICATION REPORT ===')
print(classification_report(test_bin_targets, test_bin_preds, target_names=['benign', 'adversarial']))
print('ROC-AUC:', roc_auc_score(test_bin_targets, test_probs))
print('Confusion Matrix:')
print(confusion_matrix(test_bin_targets, test_bin_preds))


In [ ]:
# 10. Checkpoint & Artifact Export
save_dir = 'checkpoints/agentshield_guard'
os.makedirs(save_dir, exist_ok=True)

torch.save(model.state_dict(), os.path.join(save_dir, 'pytorch_model.bin'))
tokenizer.save_pretrained(save_dir)

meta = {
    'model_version': 'v3.0.0-distilbert-multitask',
    'calibrated_threshold': best_tau,
    'categories': ID2CATEGORY,
    'labels': LABEL_MAP
}
with open(os.path.join(save_dir, 'model_metadata.json'), 'w') as f:
    json.dump(meta, f, indent=2)
print(f'Model weights and tokenizer saved to {save_dir}')
